## Модуль 0. Пролог: почему инфраструктура — это фундамент

### 0.1. Парадокс современного бэкенда

#### 0.1.1. Что происходит, когда сервер ждёт: блокировка, I/O и простой CPU

**Синхронная модель обработки запросов.**
В классическом синхронном сервере (например, ранний Flask под WSGI) каждый запрос обрабатывается в отдельном потоке или процессе. Поток начинает выполнение, доходит до операции, требующей внешнего ресурса — чтения из базы данных, запроса к другому сервису, записи на диск — и делает **системный вызов** (syscall), например `read()` или `recv()`.

**Состояния процесса/потока.**
Операционная система переводит поток в состояние **WAITING** (или `SLEEPING`, `D` — uninterruptible sleep). В это время:
- Поток не потребляет CPU.
- Ядро ОС ставит его в очередь ожидания события (готовности файлового дескриптора).
- **CPU простаивает** или переключается на другой поток.

**Пропускная способность ограничена задержками, а не вычислениями.**
Если обработка запроса занимает $T_{\text{cpu}} = 5$ мс вычислений и $T_{\text{io}} = 95$ мс ожидания ответа от БД, то утилизация CPU для одного потока составляет:

$$\rho_{\text{cpu}} = \frac{T_{\text{cpu}}}{T_{\text{cpu}} + T_{\text{io}}} = \frac{5}{100} = 5\%$$

Чтобы загрузить одно ядро на 100%, нужно иметь конкурентно порядка 20 таких потоков. Но каждый поток потребляет оперативную память (стек, TLS, структуры ядра) — типично 4–8 МБ на поток. 10 000 потоков = 40–80 ГБ RAM только на стеки. Это **парадокс**: сервер тратит 95% времени на ожидание, но мы вынуждены разводить тяжёлые потоки, чтобы хоть как-то загрузить железо.

#### 0.1.2. Парадокс масштабирования: 10 000 пользователей ≠ 10 000 ядер

**Конкурентность vs параллелизм.**
- **Параллелизм** (parallelism) — буквальное одновременное выполнение: $N$ ядер выполняют $N$ потоков одновременно.
- **Конкурентность** (concurrency) — способность системы обрабатывать множество задач в перекрывающихся временных интервалах. На одном ядре это достигается мультиплексированием.

**Переключение контекста (context switch).**
Когда ОС переключает CPU с одного потока на другой, она выполняет:
1. Сохранение регистров, счётчика команд (PC), указателя стека текущего потока в структуру `task_struct`.
2. Обновление таблицы страниц памяти (если процессы разные — TLB flush).
3. Загрузка состояния нового потока.
4. Переход в пространство пользователя.

Накладные расходы на переключение контекста в Linux — порядка 1–10 мкс для потоков, 1–10 мс для процессов (из-за TLB invalidation). При 10 000 потоках и частых переключениях эти накладные расходы становятся доминирующими.

**Решение: event-driven модель и неблокирующий I/O.**
Вместо одного потока на запрос используется **один поток на много запросов**:
- Сокеты переводятся в неблокирующий режим (`O_NONBLOCK`).
- Ядро уведомляет приложение о готовности через `epoll` (Linux), `kqueue` (BSD/macOS) или `IOCP` (Windows).
- Приложение в цикле обрабатывает только те соединения, у которых появились данные.
- Это решение **C10K problem** (10 000 одновременных соединений): один поток обслуживает тысячи соединений, переключаясь между ними только при наличии работы.

#### 0.1.3. Почему ML-сервисы особенно чувствительны к задержкам

**Inference — это CPU-bound или GPU-bound задача.**
В отличие от классического веб-бэкенда, где задержка определяется сетью и диском, ML-инференс — это интенсивное вычисление:
- Матричные умножения, свёртки, трансформеры.
- Время выполнения непредсказуемо: от 10 мс для маленькой модели до 30+ секунд для генеративной модели.

**Проблема GIL и event loop.**
В Python существует **Global Interpreter Lock (GIL)** — мьютекс, который позволяет только одному потоку исполнять байт-код CPython в единицу времени. Если в асинхронный event loop (asyncio) воткнуть тяжёлое CPU-вычисление (например, `model.predict()` на numpy без offload на C-библиотеки вне GIL), весь loop блокируется:
- Все остальные корутины перестают выполняться.
- Таймауты не срабатывают.
- Соединения отваливаются.

**Dependency hell в ML.**
ML-проекты требуют строгой изоляции:
- PyTorch 2.1 требует CUDA 12.1, но другой сервис использует TensorFlow, который работает только с CUDA 11.8.
- `numpy`, `scipy`, `pandas` часто имеют C-зависимости (BLAS, LAPACK), скомпилированные под конкретную архитектуру.
- «Работает на моей машине» означает: «у меня установлены те же shared libraries, те же переменные окружения и тот же GCC».

Вывод: ML-сервис нельзя просто «залить на сервер». Нужна **изоляция среды** (контейнеризация) и **вынесение тяжёлых задач** из основного потока обработки запросов.

#### 0.1.4. От монолита к распределённой системе: почему нельзя просто «вызвать функцию»

**Монолит: общая память и синхронный вызов.**
В монолите компоненты существуют в одном адресном пространстве. Компонент A вызывает функцию компонента B:
- Задержка: наносекунды (вызов в памяти).
- Семантика: синхронная, строгая типизация, общее состояние.

**Проблемы монолита при масштабировании:**
- Нельзя масштабировать отдельный компонент (например, только ML-инференс).
- Падение одного компонента (утечка памяти в модели) валит всё приложение.
- Разные команды вынуждены деплоить весь монолит целиком.

**Распределённая система: процессы общаются через сеть.**
Когда сервисы разделены, возникают два фундаментальных ограничения:

1. **Temporal coupling (временная связанность).**
   Сервис A вызывает сервис B через HTTP/gRPC. Если B недоступен (перегружен, обновляется, упал), A должен ждать или падать. Вызывающий и вызываемый должны существовать **одновременно**.

2. **Spatial coupling (пространственная связанность).**
   A должен знать адрес B (`http://ml-service:8080`). Если B переехал или масштабировался, A должен об этом узнать (Service Discovery).

**Асинхронные очереди как декомпозиция связанности.**
Вместо прямого вызова A публикует сообщение в брокер (например, Kafka или Redis). B читает сообщение когда готов:
- **Temporal decoupling:** A и B не обязаны быть активны одновременно. Брокер буферизует сообщения.
- **Spatial decoupling:** A не знает, кто и где обработает сообщение. B не знает, кто его отправил.
- **Load leveling:** брокер выступает амортизатором (shock absorber). Пиковая нагрузка сглаживается.

### 0.2. Ландшафт решений

#### 0.2.1. Виртуализация vs контейнеризация vs serverless

**Виртуализация: полная изоляция на уровне железа.**
- **Гипервизор Type 1** (bare-metal: VMware ESXi, Xen, KVM) — работает прямо на железе.
- **Гипервизор Type 2** (hosted: VirtualBox, VMware Workstation) — работает поверх хостовой ОС.
- Каждая ВМ имеет **собственное ядро ОС**, собственную файловую систему, собственные драйверы.
- Изоляция: максимальная (аппаратная).
- Накладные расходы: десятки-сотни мегабайт RAM на ВМ, секунды-минуты на загрузку.
- Плотность размещения: низкая (10–100 ВМ на сервер).

**Контейнеризация: изоляция на уровне ОС.**
- Все контейнеры делят **одно ядро хостовой ОС**.
- Изоляция обеспечивается **namespaces** (PID, NET, MNT, UTS, IPC, USER, CGROUP) и **cgroups** (ограничение ресурсов).
- Нет эмуляции железа, нет собственного ядра.
- Накладные расходы: единицы мегабайт RAM, миллисекунды на старт.
- Плотность: высокая (тысячи контейнеров на сервер).
- Идеально для микросервисов и ML-моделей с разными зависимостями.

**Serverless: функция как единица развёртывания.**
- Примеры: AWS Lambda, Google Cloud Functions, Yandex Cloud Functions.
- Разработчик загружает код; платформа управляет инфраструктурой.
- **Event-driven:** функция вызывается по событию (HTTP-запрос, сообщение из очереди, таймер).
- **Масштабирование до нуля:** если нет запросов, нет затрат (но есть cold start — задержка 100 мс–секунды при первом вызове).
- **Ограничения:** максимальное время выполнения (15 минут в Lambda), ограниченная среда (нет произвольных системных вызовов), vendor lock-in.
- Для ML: подходит для лёгкого inference, не подходит для длительного обучения или GPU-задач.

**Сравнительная таблица:**

| Критерий | Виртуализация | Контейнеризация | Serverless |
|---|---|---|---|
| Время запуска | Секунды–минуты | Миллисекунды | 100 мс–секунды (cold start) |
| Изоляция | Аппаратная | Программная (ядро общее) | Программная (песочница) |
| Плотность | Низкая | Высокая | Очень высокая |
| Контроль над ОС | Полный | Частичный | Минимальный |
| GPU-поддержка | Да (PCI-passthrough) | Да (NVIDIA Docker) | Ограниченно |
| Идеально для | Legacy, разные ОС | Микросервисы, ML | Webhooks, обработка событий |

#### 0.2.2. Синхронный RPC vs асинхронные очереди: coupling и семантика

**RPC (Remote Procedure Call): REST, gRPC, SOAP.**
- Семантика: «вызови функцию на удалённом сервере и верни результат».
- **Temporal coupling:** клиент блокируется до получения ответа. Если сервис B недоступен, A ждёт таймаута или падает.
- **Spatial coupling:** клиент знает endpoint (`host:port`) или использует Service Discovery.
- **Backpressure:** клиент сам регулирует нагрузку, но при всплеске может перегрузить сервис (thundering herd).
- **Latency:** складывается из сетевой задержки (RTT), времени обработки и очереди на стороне сервера.

**Асинхронные очереди (Message Brokers).**
- Семантика: «отправь сообщение и забудь» (fire-and-forget) или «отправь и дождись подтверждения из другого сервиса».
- **Temporal decoupling:** продюсер и консьюмер работают независимо. Сообщение хранится в брокере минуты, часы, дни.
- **Spatial decoupling:** продюсер публикует в топик/очередь; консьюмеры подписываются. Никто не знает адресов друг друга.
- **Буферизация нагрузки:** брокер выступает амортизатором. Если консьюмеры не справляются, очередь растёт, но система не падает.
- **Гарантии доставки:** at-most-once, at-least-once, exactly-once (будут разобраны в модуле 4).

**Математическая модель: брокер как система массового обслуживания.**
Брокер можно моделировать как очередь:
- $\lambda$ — интенсивность поступления сообщений (msg/s).
- $\mu$ — интенсивность обработки одним консьюмером (msg/s).
- $c$ — число консьюмеров.
- Условие стабильности: $\lambda < c \cdot \mu$.

Если $\lambda > c \cdot \mu$, очередь неограниченно растёт (если нет политики отбрасывания). Это ключевое уравнение для понимания масштабирования воркеров.

#### 0.2.3. Roadmap курса: от механики к production

Курс построен по принципу **«от фундамента к инструменту»**. Каждый инструмент обретает смысл только после понимания проблемы, которую он решает:

1. **Модули 1–2:** Механика Linux — процессы, namespaces, cgroups, OverlayFS. Почему контейнер изолирован, но лёгок.
2. **Модуль 3:** Docker и Docker Compose — инструменты для создания изолированных сред и связки сервисов.
3. **Модуль 4:** Теория распределённых систем — CAP, queuing theory, гарантии доставки. Почему нельзя просто «послать сообщение и забыть».
4. **Модули 5–6:** Redis Streams и Celery — введение в брокеры, эфемерные очереди задач, ACK/NACK.
5. **Модули 7–8:** Apache Kafka — распределённый лог, партиции, оффсеты, consumer groups, aiokafka.
6. **Модуль 9:** Масштабирование — ребалансировка, backpressure, отказоустойчивость.
7. **Модуль 10:** Идемпотентность — математическое определение, стратегии дедупликации, Saga pattern.
8. **Модуль 11:** Архитектурные отличия — когда Task Queue, когда Event Log, когда Kafka + Celery вместе.
9. **Модуль 12:** Интеграционный проект — полный пайплайн: FastAPI -> Kafka -> Celery Workers -> PostgreSQL, с idempotency, DLQ, мониторингом и chaos-тестированием.

**Ключевой принцип:** мы не изучаем Docker «потому что он модный», а потому что он решает проблему изоляции ML-зависимостей. Мы не изучаем Kafka «потому что enterprise», а потому что она решает проблему temporal coupling в распределённых ML-системах.